# Data Cleaning and Preprocessing


In [3]:
# Import required libraries
import pandas as pd
import numpy as np


# Load dataset
DATA_PATH = "weather_data.csv"
df = pd.read_csv(DATA_PATH)


# Convert DATE column to datetime format
# This ensures correct chronological ordering for time series modelling
df["DATE"] = pd.to_datetime(df["DATE"], format="%d-%m-%Y")


# Sort the dataset chronologically
# Walk-forward validation requires strict temporal ordering
df = df.sort_values("DATE")


# Set DATE as index for time-series operations
df = df.set_index("DATE")


# Drop non-informative columns
# STATION and NAME contain constant information and add no predictive value
columns_to_drop = ["STATION", "NAME"]
df = df.drop(columns=columns_to_drop)


# Identify weather event flag columns (WT01–WT22 etc.)
# In NOAA datasets:
# 1 = event occurred
# NaN = event did not occur
# Therefore NaN should be replaced with 0
wt_columns = [col for col in df.columns if col.startswith("WT")]

df[wt_columns] = df[wt_columns].fillna(0)
df[wt_columns] = df[wt_columns].astype(int)


# Remove potential target leakage
# TAVG is typically derived from TMAX and TMIN
# Using it would leak target information into the model
if "TAVG" in df.columns:
    df = df.drop(columns=["TAVG"])


# Remove sparse time-of-occurrence columns
# These columns represent time of events rather than weather magnitude
# They usually contain excessive missing values and weak predictive power
sparse_time_columns = ["FMTM", "PGTM"]

for col in sparse_time_columns:
    if col in df.columns:
        df = df.drop(columns=[col])


# Handle precipitation and snow related missing values
# In weather datasets, missing values generally mean "no event"
zero_fill_columns = ["PRCP", "SNOW", "SNWD", "WESD"]

for col in zero_fill_columns:
    if col in df.columns:
        df[col] = df[col].fillna(0)


# Forward fill remaining missing values
# This preserves temporal continuity for weather variables
df = df.fillna(method="ffill")


# Final validation checks
print("Final Dataset Shape:", df.shape)
print("Remaining Missing Values:", df.isna().sum().sum())


# Preview cleaned dataset
print(df.head())


# Save cleaned dataset for downstream feature engineering and modelling
OUTPUT_PATH = "weather_data_preprocessed.csv"
df.to_csv(OUTPUT_PATH)

Final Dataset Shape: (5844, 30)
Remaining Missing Values: 0
             AWND  PRCP  SNOW  SNWD  TMAX  TMIN   WDF2   WDF5  WESD  WSF2  \
DATE                                                                        
2010-01-01   5.37  0.04   0.0   0.0  41.0    32  320.0  320.0   0.0  19.9   
2010-01-02  23.71  0.00   0.1   0.0  33.0    17  300.0  300.0   0.0  34.9   
2010-01-03  25.50  0.00   0.0   0.0  24.0    18  310.0  320.0   0.0  40.9   
2010-01-04  17.67  0.00   0.0   0.0  32.0    20  310.0  310.0   0.0  23.9   
2010-01-05  15.88  0.00   0.0   0.0  32.0    21  310.0  310.0   0.0  21.0   

            ...  WT11  WT13  WT14  WT15  WT16  WT17  WT18  WT19  WT21  WT22  
DATE        ...                                                              
2010-01-01  ...     0     1     0     0     1     0     1     0     0     0  
2010-01-02  ...     0     0     0     0     1     0     1     0     0     0  
2010-01-03  ...     0     0     0     0     1     0     1     0     0     0  
2010-01-04

/tmp/ipykernel_504/3771154420.py:70: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method="ffill")
